In [1]:
import os
import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import scipy.stats as ss

import torch
import torch.nn as nn
from torch.optim import *
from torch.utils.data import Dataset, DataLoader
from torch.optim import *
from torch.nn import functional as F
import torchvision
from PIL import Image
from torchvision import transforms
from warnings import filterwarnings; filterwarnings("ignore")
import torch
import torch.nn as nn
import torchvision.models as models
from torch.optim import lr_scheduler
import ast


import joblib

In [2]:
class PigDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["image_id"])
        image = Image.open(img_path).convert("RGB")

        # Parse bbox string → list
        bbox = ast.literal_eval(row["bbox"])
        x, y, w, h = bbox

        # Convert center format → corner format
        x1 = int(x - w / 2)
        y1 = int(y - h / 2)
        x2 = int(x + w / 2)
        y2 = int(y + h / 2)

        crop = image.crop((x1, y1, x2, y2))

        if self.transform:
            crop = self.transform(crop)

        label = torch.tensor(int(row["class_id"]))

        return crop, label

In [3]:
class ToTensorNoNumPy:
    def __call__(self, img):
        # img is PIL.Image in RGB
        img = img.convert("RGB")
        tensor = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes()))
        tensor = tensor.view(img.size[1], img.size[0], 3)
        tensor = tensor.permute(2, 0, 1).float().div(255.0)
        return tensor

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.2       # safe
    ),
    ToTensorNoNumPy(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15))
])

train_dataset = PigDataset(
    csv_file="/kaggle/input/datasets/dataset69420/dataset/pig_posture_recognition/train.csv",
    image_dir="/kaggle/input/datasets/dataset69420/dataset/pig_posture_recognition/train_images",
    transform=train_transform
)

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

In [4]:
#For Testing Dataset
class PigTestDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["image_id"])
        image = Image.open(img_path).convert("RGB")

        # Parse bbox string
        bbox = ast.literal_eval(row["bbox"])
        x, y, w, h = bbox

        # Convert center → corner
        x1 = int(x - w / 2)
        y1 = int(y - h / 2)
        x2 = int(x + w / 2)
        y2 = int(y + h / 2)

        crop = image.crop((x1, y1, x2, y2))

        if self.transform:
            crop = self.transform(crop)

        return crop, row["row_id"]

In [5]:
test_dataset = PigTestDataset(
    csv_file="/kaggle/input/datasets/dataset69420/dataset/pig_posture_recognition/test.csv",
    image_dir="/kaggle/input/datasets/dataset69420/dataset/pig_posture_recognition/test_images",
    transform=val_transform  # use validation transform
)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


#Lets define Training Variables

In [6]:
# Cell was removed as it's no longer needed after mlflow removal.

In [7]:
def train_model(model, train_loader, val_loader, device, epochs=5):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = Adam(model.parameters(), lr=3e-4, weight_decay=1e-3)

    scheduler = lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )

    best_acc = 0

    # Start training run (mlflow removed)
    for epoch in range(epochs):
        model.train()
        total = 0
        correct = 0
        running_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            running_loss += loss.item() * labels.size(0)

        train_acc = correct / total
        train_loss = running_loss / total

        # Validation Step
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0.0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                val_loss += loss.item() * labels.size(0)

        val_acc = val_correct / val_total
        val_loss = val_loss / val_total

        # Step scheduler with validation accuracy
        scheduler.step(val_acc)

        print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Train Loss={train_loss:.4f} | Val Acc={val_acc:.4f}, Val Loss={val_loss:.4f}")

        # Save best model (mlflow removed - if model saving is desired, it needs to be implemented separately)
        if val_acc > best_acc:
            best_acc = val_acc
            # mlflow.pytorch.log_model(model, "best_model") # Removed mlflow call

    # mlflow.log_metric("best_val_acc", best_acc) # Removed mlflow call

    return best_acc

In [8]:
def create_model(model_name, num_classes=5):
    """Helper function to initialize models"""
    if model_name == "resnet18":
        model = models.resnet18(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "mobilenet_v3":
        model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights='IMAGENET1K_V1')
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "vgg16":
        model = models.vgg16(weights='IMAGENET1K_V1')
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    else:
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

# Define the models to test
models_to_test = {
    "ResNet18": "resnet18",
    "MobileNetV3": "mobilenet_v3",
    "EfficientNetB0": "efficientnet_b0"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results = {}
trained_models = {}

for name, model_id in models_to_test.items():
    print(f"\nTraining {name}...")
    model = create_model(model_id)

    best_val_acc = train_model(
        model,
        train_loader,
        val_loader,
        device,
        epochs=5
    )

    results[name] = best_val_acc
    trained_models[name] = model 
print("\nFinal Results:")
print(results)


Training ResNet18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 156MB/s] 


Epoch 1: Train Acc=0.7141, Train Loss=0.9618 | Val Acc=0.7554, Val Loss=0.8722
Epoch 2: Train Acc=0.7959, Train Loss=0.8151 | Val Acc=0.8252, Val Loss=0.7622
Epoch 3: Train Acc=0.8220, Train Loss=0.7685 | Val Acc=0.8384, Val Loss=0.7391
Epoch 4: Train Acc=0.8302, Train Loss=0.7557 | Val Acc=0.8515, Val Loss=0.7143
Epoch 5: Train Acc=0.8390, Train Loss=0.7377 | Val Acc=0.8442, Val Loss=0.7184

Training MobileNetV3...
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 98.7MB/s]


Epoch 1: Train Acc=0.7112, Train Loss=0.9576 | Val Acc=0.7663, Val Loss=0.8717
Epoch 2: Train Acc=0.8096, Train Loss=0.7828 | Val Acc=0.8349, Val Loss=0.7554
Epoch 3: Train Acc=0.8346, Train Loss=0.7324 | Val Acc=0.8318, Val Loss=0.7511
Epoch 4: Train Acc=0.8577, Train Loss=0.6959 | Val Acc=0.8230, Val Loss=0.7519
Epoch 5: Train Acc=0.8679, Train Loss=0.6754 | Val Acc=0.8285, Val Loss=0.7354

Training EfficientNetB0...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 144MB/s] 


Epoch 1: Train Acc=0.7406, Train Loss=0.9066 | Val Acc=0.8669, Val Loss=0.6815
Epoch 2: Train Acc=0.8463, Train Loss=0.7191 | Val Acc=0.8920, Val Loss=0.6268
Epoch 3: Train Acc=0.8741, Train Loss=0.6697 | Val Acc=0.9140, Val Loss=0.5902
Epoch 4: Train Acc=0.8822, Train Loss=0.6489 | Val Acc=0.9159, Val Loss=0.5820
Epoch 5: Train Acc=0.8912, Train Loss=0.6358 | Val Acc=0.9230, Val Loss=0.5793

Final Results:
{'ResNet18': 0.8514506288133483, 'MobileNetV3': 0.8348898020171834, 'EfficientNetB0': 0.923048188270452}


#Predicting The Test Set

In [13]:
models_probs = {name: [] for name in trained_models}

for model in trained_models.values():
    model.eval()

with torch.no_grad():
    for images, _ in test_loader:  
        images = images.to(device)

        for name, model in trained_models.items():
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            models_probs[name].append(probs.cpu())

for name in models_probs:
    models_probs[name] = torch.cat(models_probs[name], dim=0)

eps = 1e-9
log_probs = 0

acc = np.array([0.85145, 0.83488, 0.92304])

weights = np.exp(acc * 5)   # temperature scaling
weights = weights / weights.sum()

ensemble_weights = dict(zip(
    ['ResNet18', 'MobileNetV3', 'EfficientNetB0'],
    weights
))

for name, probs in models_probs.items():
    weight = ensemble_weights[name]
    log_probs += weight * torch.log(probs + eps)

ensemble_probs_geo = torch.exp(log_probs)
ensemble_preds_geo = torch.argmax(ensemble_probs_geo, dim=1).numpy()

In [15]:
#Ensembling the predictions
#Formula goes like this: alpha * (1 - alpha) * probs (of each model)
test_ids = pd.read_csv("/kaggle/input/datasets/dataset69420/dataset/pig_posture_recognition/test.csv")
test_ids = test_ids["row_id"]


submission = pd.DataFrame({
    "row_id": test_ids,
    "class_id": ensemble_preds_geo
})

submission.to_csv("Submission_prediction.csv", index=False)